# Radeon-Assistant 启动指南

这是 Radeon-Assistant 的启动 Notebook，包含环境配置和应用启动步骤。

**技术栈**: vLLM + ROCm + Qwen2.5-14B-Instruct + AMD Radeon PRO W7900

## 第一步：安装依赖

安装基础依赖 + vLLM (ROCm 预编译 wheel)

In [ ]:
import subprocess
result = subprocess.run(['pip', 'install', '-r', '../requirements.txt'], capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)

In [ ]:
# 安装 vLLM ROCm wheel
import subprocess
result = subprocess.run(
    ['pip', 'install', 'vllm', '--extra-index-url', 'https://wheels.vllm.ai/rocm/', '--no-cache-dir'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)

## 第二步：检查 GPU 环境

确认 ROCm 和 W7900 GPU 可用

In [ ]:
import subprocess
result = subprocess.run(['rocm-smi', '--showproductname'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
# 验证 PyTorch ROCm
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'
os.environ['PYTORCH_ROCM_ARCH'] = 'gfx1100'
os.environ['HIP_VISIBLE_DEVICES'] = '0'

import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'ROCm 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 第三步：下载模型（首次运行）

下载 Qwen2.5-14B-Instruct (safetensors 格式，约 30GB)

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../scripts/download_model.py', '--model', 'qwen2.5-14b'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
print(result.stderr[-1000:] if len(result.stderr) > 1000 else result.stderr)

## 第四步：启动 Web 应用

启动 Streamlit Web UI（后台运行）

In [ ]:
import subprocess
import threading
import time

def run_streamlit():
    env = os.environ.copy()
    env['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'
    env['HIP_VISIBLE_DEVICES'] = '0'
    result = subprocess.run(
        ['streamlit', 'run', '../ui/web_app.py', '--server.port', '7860', '--server.address', '0.0.0.0', '--server.headless', 'true'],
        capture_output=True, text=True, env=env
    )
    print(result.stdout)
    print(result.stderr)

thread = threading.Thread(target=run_streamlit)
thread.start()

print('等待应用启动...')
time.sleep(5)
print('应用已启动！访问 http://localhost:7860 或 JupyterLab 的 /proxy/7860/')

## 第五步：测试推理引擎

直接在 Notebook 里测试 vLLM 推理

In [ ]:
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'
os.environ['PYTORCH_ROCM_ARCH'] = 'gfx1100'
os.environ['HIP_VISIBLE_DEVICES'] = '0'

import sys
sys.path.insert(0, '..')

import yaml
from inference.engine import InferenceEngine, InferenceConfig

with open('../config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

model_config = config.get('model', {})
inference_config = InferenceConfig(
    model_path=model_config.get('path', './models/Qwen2.5-14B-Instruct'),
    n_ctx=model_config.get('n_ctx', 8192),
    temperature=model_config.get('temperature', 0.7),
    max_tokens=model_config.get('max_tokens', 4096),
    dtype=model_config.get('dtype', 'float16'),
    gpu_memory_utilization=model_config.get('gpu_memory_utilization', 0.85),
    tensor_parallel_size=model_config.get('tensor_parallel_size', 1),
)

print('正在加载模型（首次约 2-3 分钟）...')
engine = InferenceEngine(inference_config)
print('模型加载成功')
print(engine.get_model_info())

In [ ]:
# 测试对话
response = engine.chat_completion([
    {'role': 'user', 'content': '你好，用 100 字介绍一下你自己'}
])
print('回复:', response)

In [ ]:
# 测试性能
import time
t0 = time.time()
response = engine.generate('请用 100 字介绍 AMD Radeon GPU', max_tokens=100)
t1 = time.time()
print(f'生成内容: {response}')
print(f'\n--- 性能数据 ---')
print(f'总耗时: {t1-t0:.2f}s')
print(f'生成速度: {100/(t1-t0):.1f} tokens/s')

## 停止应用

In [ ]:
import subprocess
result = subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True, text=True)
print('应用已停止')